<a href="https://colab.research.google.com/github/yhnx/EN4554_Assignment_2/blob/main/Assigment_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import os, math, time, random, copy
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision
from torch.optim import Optimizer
from IPython.display import display

In [5]:
seed = 42

random.seed(seed); np.random.seed(seed)
torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.backends.cudnn.benchmark = True
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
print('device:', device, '| torch', torch.__version__)

device: cpu | torch 2.11.0+cpu


In [8]:
# ---- Load CIFAR-10, split 45k/5k, normalise with TRAIN statistics ----
def load_cifar10(root='./data'):
    tr = torchvision.datasets.CIFAR10(root, train=True,  download=True)
    te = torchvision.datasets.CIFAR10(root, train=False, download=True)
    f = lambda a: torch.from_numpy(a).permute(0, 3, 1, 2).contiguous()      # uint8, N x 3 x 32 x 32
    return f(tr.data), torch.tensor(tr.targets), f(te.data), torch.tensor(te.targets)

CLASSES = ['plane','car','bird','cat','deer','dog','frog','horse','ship','truck']
X_all, Y_all, X_te, Y_te = load_cifar10()

g = torch.Generator().manual_seed(seed)
perm = torch.randperm(len(X_all), generator=g)
tr_idx, va_idx = perm[:45000], perm[45000:]


x01 = X_all[tr_idx].float() / 255
MEAN = x01.mean((0, 2, 3), keepdim=True)          # statistics from the training split only
STD  = x01.std((0, 2, 3), keepdim=True)
norm = lambda x: (x.float() / 255 - MEAN) / STD

Xtr, Ytr = norm(X_all[tr_idx]).to(device), Y_all[tr_idx].to(device)
Xva, Yva = norm(X_all[va_idx]).to(device), Y_all[va_idx].to(device)
Xte, Yte = norm(X_te), Y_te                        # kept on CPU, NOT used until Task 4
del x01
print('train', tuple(Xtr.shape), '| val', tuple(Xva.shape), '| mean', MEAN.flatten().tolist(), '| std', STD.flatten().tolist())

fig, axs = plt.subplots(2, 8, figsize=(12, 3.4))
for a, i in zip(axs.flat, range(16)):
    img = (Xtr[i].cpu() * STD[0] + MEAN[0]).clamp(0, 1).permute(1, 2, 0)
    a.imshow(img); a.set_title(CLASSES[Ytr[i].item()], fontsize=8); a.axis('off')
plt.tight_layout(); plt.show()

KeyboardInterrupt: 

In [ ]:
# ---- Shared utilities: evaluation, GPU augmentations (cutout / mixup written by hand), training loop ----
def amp_ctx(on):
    return torch.autocast(device_type=device, dtype=torch.float16, enabled=(on and device == 'cuda'))

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

@torch.no_grad()
def evaluate(model, X, Y, bs=1000, amp=False):
    model.eval(); loss, correct = 0., 0
    for i in range(0, len(X), bs):
        with amp_ctx(amp):
            out = model(X[i:i+bs])
        loss += F.cross_entropy(out.float(), Y[i:i+bs], reduction='sum').item()
        correct += (out.argmax(1) == Y[i:i+bs]).sum().item()
    return loss / len(X), correct / len(X)

def random_flip_crop(x, pad=4):
    """Random horizontal flip + random 32x32 crop from the zero-padded image (per-sample, vectorised)."""
    N, C, H, W = x.shape; d = x.device
    flip = torch.rand(N, device=d) < 0.5
    x = torch.where(flip[:, None, None, None], x.flip(3), x)
    xp = F.pad(x, (pad, pad, pad, pad))
    oy = torch.randint(0, 2 * pad + 1, (N,), device=d); ox = torch.randint(0, 2 * pad + 1, (N,), device=d)
    ar = torch.arange(H, device=d)
    rows = (oy[:, None] + ar)[:, None, :, None]
    cols = (ox[:, None] + ar)[:, None, None, :]
    n = torch.arange(N, device=d)[:, None, None, None]; c = torch.arange(C, device=d)[None, :, None, None]
    return xp[n, c, rows, cols]

def cutout(x, size=16):
    """Cutout: zero (= the mean colour after normalisation) a size x size square at a random centre; may be clipped at borders."""
    N, C, H, W = x.shape; d = x.device
    cy = torch.randint(0, H, (N, 1), device=d); cx = torch.randint(0, W, (N, 1), device=d)
    ar = torch.arange(H, device=d)[None]
    my = (ar >= cy - size // 2) & (ar < cy + size // 2)
    mx = (ar >= cx - size // 2) & (ar < cx + size // 2)
    mask = (my[:, :, None] & mx[:, None, :])[:, None]
    return x.masked_fill(mask, 0.)

def mixup_batch(x, y, alpha=0.4):
    """Mixup: x~ = lam*x_i + (1-lam)*x_j, lam ~ Beta(alpha, alpha). Loss = lam*CE(y_i) + (1-lam)*CE(y_j)."""
    lam = float(np.random.beta(alpha, alpha))
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def warmup_cosine(opt, total_steps, warmup_steps):
    def f(s):
        if s < warmup_steps: return (s + 1) / warmup_steps
        p = (s - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1 + math.cos(math.pi * p))
    return torch.optim.lr_scheduler.LambdaLR(opt, f)

def train(model, opt, X, Y, Xv, Yv, epochs, bs=128, flipcrop=False, cut=False, mix=False,
          sched=None, label_smoothing=0.0, amp=False, eval_train=False, verbose=True, name=''):
    """Generic training loop. Returns history dict. If eval_train, train accuracy is measured on the clean
    (un-augmented) training set in eval mode; otherwise it is the running accuracy during the epoch."""
    scaler = torch.amp.GradScaler(device, enabled=(amp and device == 'cuda'))
    hist = {k: [] for k in ['train_loss', 'train_acc', 'val_loss', 'val_acc']}; hist['diverged'] = False
    nb = max(1, len(X) // bs)
    for e in range(epochs):
        model.train(); t0 = time.time()
        perm = torch.randperm(len(X), device=X.device)
        run_loss = torch.zeros((), device=X.device); run_corr = torch.zeros((), device=X.device); seen = 0
        for b in range(nb):
            idx = perm[b * bs:(b + 1) * bs]
            xb, yb = X[idx], Y[idx]
            if flipcrop: xb = random_flip_crop(xb)
            if cut:      xb = cutout(xb)
            with amp_ctx(amp):
                if mix:
                    xb, ya, yb2, lam = mixup_batch(xb, yb)
                    out = model(xb)
                    loss = lam * F.cross_entropy(out, ya, label_smoothing=label_smoothing) \
                         + (1 - lam) * F.cross_entropy(out, yb2, label_smoothing=label_smoothing)
                    tgt = ya
                else:
                    out = model(xb); loss = F.cross_entropy(out, yb, label_smoothing=label_smoothing); tgt = yb
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            if sched is not None: sched.step()
            run_loss += loss.detach() * len(idx); run_corr += (out.argmax(1) == tgt).sum(); seen += len(idx)
        tl = (run_loss / seen).item()
        if not math.isfinite(tl):                       # training blew up
            hist['diverged'] = True
            hist['train_loss'].append(float('nan')); hist['train_acc'].append(0.1)
            hist['val_loss'].append(float('nan')); hist['val_acc'].append(0.1)
            if verbose: print(f'{name} epoch {e+1}: DIVERGED');
            break
        vl, va = evaluate(model, Xv, Yv, amp=amp)
        ta = evaluate(model, X, Y, amp=amp)[1] if eval_train else (run_corr / seen).item()
        for k, v in zip(['train_loss', 'train_acc', 'val_loss', 'val_acc'], [tl, ta, vl, va]): hist[k].append(v)
        if verbose and (e % max(1, epochs // 10) == 0 or e == epochs - 1):
            print(f'{name} ep {e+1:3d}/{epochs} | train loss {tl:.3f} acc {ta:.3f} | val loss {vl:.3f} acc {va:.3f} | {time.time()-t0:.1f}s')
    return hist